In [ ]:
%pip -q install pydicom scikit-image scikit-learn tqdm
%pip install -q pydicom highdicom scikit-image numpy requests



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 68.0 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path

BASE_PATH  = Path("/content/drive/MyDrive/lidc/LIDC-IDRI")
DICOM_ROOT = BASE_PATH

print("BASE_PATH:", BASE_PATH.exists(), BASE_PATH)
print("DICOM_ROOT:", DICOM_ROOT.exists(), DICOM_ROOT)


Mounted at /content/drive
BASE_PATH: True /content/drive/MyDrive/lidc/LIDC-IDRI
DICOM_ROOT: True /content/drive/MyDrive/lidc/LIDC-IDRI


In [ ]:
# === Rutas en Drive y reparación de índices ===
from pathlib import Path
import pandas as pd

BASE_PATH = Path("/content/drive/MyDrive/lidc")
DICOM_ROOT = BASE_PATH / "LIDC-IDRI"

print("BASE_PATH:", BASE_PATH.exists(), BASE_PATH)
print("DICOM_ROOT:", DICOM_ROOT.exists(), DICOM_ROOT)

# Repara rutas antiguas en dataset_index/resolved_index si existen
for name in ["dataset_index.csv", "resolved_index.csv"]:
    p = BASE_PATH / name
    if p.exists():
        df = pd.read_csv(p)
        for col in ["example_dicom_path", "series_dir"]:
            if col in df.columns:
                df[col] = df[col].astype(str).str.replace(
                    "/content/lidc_remote", str(BASE_PATH), regex=False
                )
        df.to_csv(p, index=False)
        print(f"✔️ Reparado {name}")
    else:
        print(f"(aviso) No existe {name}, se creará si es necesario.")


BASE_PATH: True /content/drive/MyDrive/lidc
DICOM_ROOT: True /content/drive/MyDrive/lidc/LIDC-IDRI
✔️ Reparado dataset_index.csv
✔️ Reparado resolved_index.csv


In [ ]:
import os, math, numpy as np, pydicom
from pathlib import Path
from skimage.filters import gaussian
from skimage.morphology import remove_small_objects, binary_closing, disk
from skimage.measure import label, regionprops
from skimage.segmentation import clear_border

def read_series_sorted(series_dir: str):
    """Devuelve lista de datasets pydicom de la serie ordenados por posición/instancia."""
    sdir = Path(series_dir)
    dsets = []
    for fp in sorted(sdir.glob("*.dcm")):
        try:
            ds = pydicom.dcmread(str(fp))
            dsets.append(ds)
        except:
            pass
    def zkey(ds):
        ipp = getattr(ds, "ImagePositionPatient", None)
        if ipp is not None and len(ipp) >= 3:
            try: return float(ipp[2])
            except: pass
        return float(getattr(ds, "InstanceNumber", 0))
    dsets.sort(key=zkey)
    return dsets

def to_hu(ds):
    arr = ds.pixel_array.astype(np.int16, copy=False)
    slope = float(getattr(ds, "RescaleSlope", 1.0))
    inter = float(getattr(ds, "RescaleIntercept", 0.0))
    return arr * slope + inter

def lung_window(img_hu, center=-600.0, width=1500.0):
    lo, hi = center - width/2.0, center + width/2.0
    img = np.clip(img_hu, lo, hi)
    return (img - lo) / (hi - lo + 1e-6)

def get_spacing(ds):
    """(row_spacing_mm, col_spacing_mm, slice_thickness_mm)"""
    ps = getattr(ds, "PixelSpacing", [1.0, 1.0])
    row_mm, col_mm = float(ps[0]), float(ps[1])
    st = float(getattr(ds, "SliceThickness", getattr(ds, "SpacingBetweenSlices", 1.0)))
    return row_mm, col_mm, st


Snippet

In [ ]:
import pandas as pd

# Carga el índice ya parcheado
ridx = pd.read_csv(BASE_PATH / "resolved_index.csv")

# Filtra los que tienen etiqueta 1 (maligno/sospechoso)
ridx_malignos = ridx.query("label == 1").copy()

# Ordena por número de slices (opcional)
ridx_malignos = ridx_malignos.sort_values("num_slices", ascending=False)

print(f"Pacientes con nódulos >3 mm o sospechosos: {len(ridx_malignos)}\n")
print(ridx_malignos[["patient_id", "num_slices", "series_dir"]].head(15))


Pacientes con nódulos >3 mm o sospechosos: 94

          patient_id  num_slices  \
991   LIDC-IDRI-0994         545   
1001  LIDC-IDRI-1004         529   
541   LIDC-IDRI-0543         525   
936   LIDC-IDRI-0939         465   
383   LIDC-IDRI-0385         445   
808   LIDC-IDRI-0811         371   
918   LIDC-IDRI-0921         350   
921   LIDC-IDRI-0924         335   
962   LIDC-IDRI-0965         328   
999   LIDC-IDRI-1002         321   
71    LIDC-IDRI-0072         305   
566   LIDC-IDRI-0568         295   
890   LIDC-IDRI-0893         290   
491   LIDC-IDRI-0493         285   
763   LIDC-IDRI-0766         280   

                                             series_dir  
991   /content/drive/MyDrive/lidc/LIDC-IDRI/LIDC-IDR...  
1001  /content/drive/MyDrive/lidc/LIDC-IDRI/LIDC-IDR...  
541   /content/drive/MyDrive/lidc/LIDC-IDRI/LIDC-IDR...  
936   /content/drive/MyDrive/lidc/LIDC-IDRI/LIDC-IDR...  
383   /content/drive/MyDrive/lidc/LIDC-IDRI/LIDC-IDR...  
808   /content/drive/MyDrive

Detector clásico

In [ ]:
from skimage.filters import threshold_otsu

def area_px_min_for_diam_mm(diam_mm, row_mm, col_mm):
    # área circular aproximada: pi * r^2, con r_px = (diam_mm/2) / mm_per_px
    mm_per_px = math.sqrt(row_mm * col_mm)  # promedio geométrico
    r_px = (diam_mm / 2.0) / mm_per_px
    return math.pi * (r_px**2)

def detect_nodules_slice(img_hu, row_mm, col_mm, min_diam_mm=3.0):
    from skimage.morphology import disk as morph_disk  # 👈 nombre seguro
    # 1) ventana y suavizado
    img = lung_window(img_hu)
    img = gaussian(img, sigma=1.0, preserve_range=True)

    # 2) umbral
    inv = 1.0 - img
    t = threshold_otsu(inv)
    mask = inv > t

    # 3) morfología
    mask = binary_closing(mask, footprint=morph_disk(1))
    mask = clear_border(mask)

    # 4) quitar objetos chicos (≥ 3 mm)
    a_min = area_px_min_for_diam_mm(min_diam_mm, row_mm, col_mm)
    mask = remove_small_objects(mask, min_size=int(max(1, a_min)))

    return mask.astype(bool)

Detección 3D

In [ ]:
def detect_volume(series_dir: str, min_diam_mm=3.0, min_slices_touch=1):
    """Devuelve (mask_3d [Z,H,W] bool, dsets_ordenados, spacing)."""
    dsets = read_series_sorted(series_dir)
    assert len(dsets) > 0, f"Sin DICOMs en {series_dir}"
    row_mm, col_mm, st_mm = get_spacing(dsets[0])

    masks = []
    for ds in dsets:
        hu = to_hu(ds)
        m = detect_nodules_slice(hu, row_mm, col_mm, min_diam_mm=min_diam_mm)
        masks.append(m)
    vol = np.stack(masks, axis=0)  # (Z,H,W) bool

    # (Opcional) filtrar objetos 3D que aparezcan en menos de N slices
    if min_slices_touch > 1:
        lab = label(vol)
        keep = np.zeros_like(vol, dtype=bool)
        for reg in regionprops(lab):
            z, y, x = np.where(lab == reg.label)
            if len(np.unique(z)) >= min_slices_touch:
                keep[z, y, x] = True
        vol = keep

    return vol, dsets, (row_mm, col_mm, st_mm)


In [ ]:
%pip install -U highdicom


In [ ]:
# ok con cualquiera de estas ramas 0.18–0.23:
%pip install -qU highdicom>=0.18,<0.24 pydicom numpy


/bin/bash: line 1: 0.24: No such file or directory


In [ ]:
from highdicom.seg import Segmentation, SegmentDescription
from highdicom.sr.coding import CodedConcept
from highdicom.seg.enum import SegmentAlgorithmTypeValues
from pydicom.uid import generate_uid

def make_dicom_seg(mask_3d, dsets, label="Nodule >3mm"):
    # --- Códigos de propiedad (Categoría y Tipo) en CodedConcept ---
    # Category: "Morphologically Altered Structure" (SCT 49755003) o "Finding" (SCT 404684003)
    # Type:     "Nodule" (SRT G-A438)
    category_code = CodedConcept("49755003", "SCT", "Morphologically Altered Structure")
    type_code     = CodedConcept("G-A438",   "SRT", "Nodule")

    # --- SegmentDescription: usa MANUAL y NO pases algorithm_identification ---
    seg_desc = SegmentDescription(
        segment_number=1,
        segment_label=label,
        segmented_property_category=category_code,  # CodedConcept
        segmented_property_type=type_code,          # CodedConcept
        algorithm_type=SegmentAlgorithmTypeValues.MANUAL  # <- clave para no requerir AlgorithmIdentification
    )

    # --- Pixel data en el orden que espera tu highdicom: (frames, rows, cols, segments) ---
    pixel_array = (mask_3d.astype('uint8') > 0)[:, :, :, np.newaxis]  # (Z,H,W,1)

    # --- Identificadores mínimos (todos STR) ---
    study_uid  = str(dsets[0].StudyInstanceUID)
    series_uid = generate_uid()
    sop_uid    = generate_uid()

    seg = Segmentation(
        source_images=dsets,               # lista de datasets CT (en orden)
        pixel_array=pixel_array,           # (Z,H,W,1)
        segmentation_type="BINARY",        # o SegmentationTypeValues.BINARY.value
        segment_descriptions=[seg_desc],
        series_instance_uid=series_uid,
        series_number=1001,
        sop_instance_uid=sop_uid,
        instance_number=1,
        manufacturer="PACS-AI",                  # TODO: str
        manufacturer_model_name="NoduleMasker",  # TODO: str
        software_versions="0.1",                 # TODO: str
        device_serial_number="SN-0001"           # TODO: str
    )

    seg.SeriesDescription = "AI Seg: Nodules >3mm"
    return seg


In [ ]:
def save_seg(seg, out_path):
    out_path = str(out_path)
    seg.save_as(out_path)
    return out_path


In [ ]:
series_dir = "/content/drive/MyDrive/lidc/LIDC-IDRI/LIDC-IDRI-0068/01-01-2000-NA-CT CHEST W CONT-80168/4.000000-Recon 3 CT CHEST-26125"
dsets = read_series_sorted(series_dir)
Z, H, W = len(dsets), int(dsets[0].Rows), int(dsets[0].Columns)

mask = np.zeros((Z, H, W), dtype=np.uint8)
z0, z1 = Z//2 - 2, Z//2 + 3
Y, X = np.ogrid[:H, :W]
cy, cx, r = H//2, W//2, min(H,W)//12
disk = ((Y - cy)**2 + (X - cx)**2) <= r*r
mask[z0:z1] = disk

seg = make_dicom_seg(mask, dsets, label="Nodule >3mm")
out_seg = save_seg(seg, "/content/drive/MyDrive/lidc/segs/LIDC-IDRI-0068_SEG.dcm")
print("SEG guardado:", out_seg)


SEG guardado: /content/drive/MyDrive/lidc/segs/LIDC-IDRI-0068_SEG.dcm


In [ ]:
import sys, highdicom
print(sys.version)
print(highdicom.__version__)


3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
0.26.1


In [ ]:
import random, math, os
import numpy as np
import pandas as pd
from skimage.measure import label as label3d

SEGS_DIR = BASE_PATH / "segs"
SEGS_DIR.mkdir(parents=True, exist_ok=True)

# === 1) Cargar índice con etiquetas reales ===
ridx = pd.read_csv(BASE_PATH / "resolved_index.csv")

# Filtrado básico (asegura paths y tamaño mínimo)
ridx = ridx.dropna(subset=["series_dir", "label"]).copy()
ridx = ridx[ridx["num_slices"].fillna(0) > 20].copy()

# Map a texto
LABEL_MAP = {0: "Benigno", 1: "Maligno"}
ridx["diagnosis_txt"] = ridx["label"].map(LABEL_MAP)

# === 2) Muestra balanceada de 10 pacientes (si es posible 5/5) ===
N_TOTAL = 10
N_PER_CLASS = N_TOTAL // 2

ben_pool = ridx[ridx["label"] == 0]
mal_pool = ridx[ridx["label"] == 1]

take_ben = min(len(ben_pool), N_PER_CLASS)
take_mal = min(len(mal_pool), N_PER_CLASS)

# si falta en alguna clase, completa con la otra
if take_ben + take_mal < N_TOTAL:
    faltan = N_TOTAL - (take_ben + take_mal)
    if len(ben_pool) - take_ben >= faltan:
        take_ben += faltan
    else:
        take_mal += faltan

sample_df = pd.concat([
    ben_pool.sample(take_ben, random_state=42),
    mal_pool.sample(take_mal, random_state=42),
]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"➡️ Pacientes elegidos: {len(sample_df)} "
      f"({(sample_df['label']==0).sum()} benignos / {(sample_df['label']==1).sum()} malignos)")

# === 3) Procesar cada serie, detectar nódulos y exportar SEG ===
rows = []
for _, row in sample_df.iterrows():
    pid  = row["patient_id"]
    sdir = row["series_dir"]
    diag = row["diagnosis_txt"]

    print(f"\n🔎 Procesando {pid} ({diag}) …")
    try:
        # Detección 3D (>3mm). Ajusta min_slices_touch si quieres más robustez.
        vol, dsets, spacing = detect_volume(sdir, min_diam_mm=3.0, min_slices_touch=1)

        # Contar nódulos (componentes 3D)
        lab = label3d(vol)
        n_nodulos = int(lab.max())

        # Incrustar la etiqueta en la descripción de la serie SEG
        seg = make_dicom_seg(vol, dsets, label=f"Nodules >3mm ({diag})")
        seg.SeriesDescription = f"AI Seg: Nodules >3mm | Dx={diag}"

        out_path = SEGS_DIR / f"{pid}_SEG.dcm"
        save_seg(seg, out_path)

        study_uid = str(dsets[0].StudyInstanceUID)

        rows.append({
            "patient_id": pid,
            "diagnosis": diag,                 # Benigno/Maligno (real del dataset)
            "num_slices": len(dsets),
            "num_nodules_detected": n_nodulos,
            "study_instance_uid": study_uid,
            "series_dir": sdir,
            "seg_path": str(out_path),
        })
        print(f"✅ {pid}: {n_nodulos} nódulo(s) | SEG → {out_path.name}")

    except Exception as e:
        print(f"⚠️ {pid} falló: {e}")

resumen = pd.DataFrame(rows)
csv_out = SEGS_DIR / "resumen_demo.csv"
resumen.to_csv(csv_out, index=False)
print("\n📄 Resumen guardado en:", csv_out)

print("\nTip OHIF:")
for uid in resumen["study_instance_uid"].head(3):
    print(f"  /ohif/viewer?StudyInstanceUIDs={uid}&configUrl=http://localhost:8000/static/config-pacsai.js")


➡️ Pacientes elegidos: 10 (5 benignos / 5 malignos)

🔎 Procesando LIDC-IDRI-0493 (Maligno) …
✅ LIDC-IDRI-0493: 17 nódulo(s) | SEG → LIDC-IDRI-0493_SEG.dcm

🔎 Procesando LIDC-IDRI-0510 (Benigno) …
✅ LIDC-IDRI-0510: 16 nódulo(s) | SEG → LIDC-IDRI-0510_SEG.dcm

🔎 Procesando LIDC-IDRI-0211 (Maligno) …
✅ LIDC-IDRI-0211: 47 nódulo(s) | SEG → LIDC-IDRI-0211_SEG.dcm

🔎 Procesando LIDC-IDRI-0818 (Benigno) …
✅ LIDC-IDRI-0818: 63 nódulo(s) | SEG → LIDC-IDRI-0818_SEG.dcm

🔎 Procesando LIDC-IDRI-0260 (Maligno) …
✅ LIDC-IDRI-0260: 29 nódulo(s) | SEG → LIDC-IDRI-0260_SEG.dcm

🔎 Procesando LIDC-IDRI-0149 (Benigno) …
✅ LIDC-IDRI-0149: 83 nódulo(s) | SEG → LIDC-IDRI-0149_SEG.dcm

🔎 Procesando LIDC-IDRI-0068 (Maligno) …
✅ LIDC-IDRI-0068: 35 nódulo(s) | SEG → LIDC-IDRI-0068_SEG.dcm

🔎 Procesando LIDC-IDRI-0173 (Benigno) …
✅ LIDC-IDRI-0173: 35 nódulo(s) | SEG → LIDC-IDRI-0173_SEG.dcm

🔎 Procesando LIDC-IDRI-0276 (Benigno) …
✅ LIDC-IDRI-0276: 84 nódulo(s) | SEG → LIDC-IDRI-0276_SEG.dcm

🔎 Procesando LIDC-ID

In [ ]:
# si sigues teniendo el var "disk" en memoria, bórrala:
try:
    del disk
except NameError:
    pass
